## PRE-TRAINING LLM 

> Load Data

In [1]:
with open("C:/Users/ASUS/Documents/GitHub/LLM-from-Scratch/J. K. Rowling - Harry Potter 1 - Sorcerer's Stone.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:49])

Total number of character: 439742
Harry Potter and the Sorcerer's Stone


CHAPTER O


> 1. Tokenization

In [2]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|-|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:9])

103826
['Harry', 'Potter', 'and', 'the', 'Sorcerer', "'", 's', 'Stone', 'CHAPTER']


In [3]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab = {token:integer for integer,token in enumerate(all_words)}

print(vocab_size)

6667


> 1.1. TokenizerV1 (word-based tokenizer without out of vocabulary handling)

In [4]:
import re

class SimpleTokenizerV1:
    # Main function of the tokenizer
    def __init__(self, vocab):
        self.str_to_int = vocab # Create a mapping from string to integer (encoding)
        self.int_to_str = {i:s for s,i in vocab.items()} # Inverse mapping from integer to string (decoding)
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|-|\s)', text) # Split the text into tokens based on the specified delimiters
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ] # Remove leading and trailing whitespace from each token and filter out empty tokens
        ids = [self.str_to_int[s] for s in preprocessed] # Convert each token to its corresponding integer ID using the mapping
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # Convert each integer ID back to its corresponding token
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

An error occured because of Out of Vocabulary problem (Lowokwaru isn't part of the vocabulary)

In [5]:
tokenizer1 = SimpleTokenizerV1(vocab)

text = "Hermione came from Lowokwaru"
tokenizer1.encode(text)

KeyError: 'Lowokwaru'

> 1.2. TokenizerV2 (word-based tokenizer with out of vocabulary handling)

- Step 1: Replace unknown words by <|unk|> tokens
    
- Step 2: Replace spaces before the specified punctuations

In [6]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [7]:
class SimpleTokenizerV2:
    # Main function of the tokenizer
    def __init__(self, vocab):
        self.str_to_int = vocab # Create a mapping from string to integer (encoding)
        self.int_to_str = { i:s for s,i in vocab.items()} # Inverse mapping from integer to string (decoding)
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # Split the text into tokens based on the specified delimiters, including double hyphens
        preprocessed = [item.strip() for item in preprocessed if item.strip()] # Remove leading and trailing whitespace from each token and filter out empty tokens
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ] # Replace tokens that are not in the vocabulary with a special token "<|unk|>"

        ids = [self.str_to_int[s] for s in preprocessed] # Convert each token to its corresponding integer ID using the mapping
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # Convert each integer ID back to its corresponding token
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Special context token added to handle out of vocabulary problem

In [8]:
tokenizer2 = SimpleTokenizerV2(vocab)

text = "Hermione came from Lowokwaru"

tokenizer2.decode(tokenizer2.encode(text))

'Hermione came from <|unk|>'

> 1.3. Sub words-based Tokenizer (BPE Tokenization using tiktoken library)

In [9]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.11.0


In [10]:
tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

116725


BPE can tokenize "Lowokwaru" without a problem because it's a sub words-based tokenizer and somehow has find a way to construct the word "Lowokwaru"

In [11]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hermione came from Lowokwaru"

tokenizer.decode(tokenizer.encode(text))

'Hermione came from Lowokwaru'

In [12]:
# Let's check which token are used

integers = tokenizer.encode(text)
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[48523, 7935, 1625, 422, 7754, 482, 5767, 84]
Hermione came from Lowokwaru


> 2. CREATING INPUT-TARGET PAIRS

In [13]:
# Let's check the number of tokens in the whole text
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

116725


In [14]:
context_size = 4 #length of the input
#The context_size of 4 means that the model is trained to look at a sequence of 4 words (or tokens) 
#to predict the next word in the sequence. 

x = enc_text[:context_size]
y = enc_text[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [18308, 14179, 290, 262]
y:      [14179, 290, 262, 30467]


In [15]:
for i in range(1, context_size+1):
    context = enc_text[:i]
    desired = enc_text[i]

    print(context, "---->", desired)

[18308] ----> 14179
[18308, 14179] ----> 290
[18308, 14179, 290] ----> 262
[18308, 14179, 290, 262] ----> 30467


The model trained to predict the next word after 4 input word

In [16]:
for i in range(1, context_size+1):
    context = enc_text[:i]
    desired = enc_text[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

Harry ---->  Potter
Harry Potter ---->  and
Harry Potter and ---->  the
Harry Potter and the ---->  Sorcerer


>3. IMPLEMENTING DATA LOADER (for easier parallel computing using PyTorch Dataset and DataLoader)

- Step 1: Tokenize the entire text   
- Step 2: Use a sliding window to chunk the book into overlapping sequences of max_length
- Step 3: Return the total number of rows in the dataset
- Step 4: Return a single row from the dataset

In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

The following code will use the GPTDatasetV1 to load the inputs in batches via a PyTorch
- Step 1: Initialize the tokenizer
- Step 2: Create dataset
- Step 3: drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes
during training
- Step 4: The number of CPU processes to use for preprocessing

In [18]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Output show the Input-Target pairs of the first batch. With batch_size=1, hence the Loader gives only one row of Input-Target pair.

Bigger batch_size means faster computing and more stable parameters update. Here we uses batch_size=1 to illustrate how the DataLoader works.

In [20]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[18308, 14179,   290,   262]]), tensor([[14179,   290,   262, 30467]])]


With stride=1 it shows that the Input-Target pairs shifted by 1 index per batch. This is how the Sliding Window approach used to build a Input-Target pairs for the training data

In [21]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[14179,   290,   262, 30467]]), tensor([[  290,   262, 30467,   338]])]


For the real case uses we can use bigger batch_size, max_length, and stride. For minimizing data overlap and overfit risks we can use stride=max_length

In [23]:
dataloader = create_dataloader_v1(raw_text, batch_size=16, max_length=8, stride=8, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[18308, 14179,   290,   262, 30467,   338,  8026,   628],
        [  198, 41481, 16329,   198,   198, 10970, 16494,    56],
        [19494,   406,  3824,  1961,   198,   198,  5246,    13],
        [  290,  9074,    13,   360,  1834,  1636,    11,   286],
        [ 1271,  1440,    11,  4389, 16809,  9974,    11,   547],
        [ 6613,   284,   910,   198,  5562,   484,   547,  7138],
        [ 3487,    11,  5875,   345,   845,   881,    13,  1119],
        [  547,   262,   938,   198, 15332,   345,  1549,  1607],
        [  284,   307,  2950,   287,  1997,  6283,   393, 11428],
        [   11,   198, 13893,   484,   655,  1422,   470,  1745],
        [  351,   884, 18149,    13,   198,   198,  5246,    13],
        [  360,  1834,  1636,   373,   262,  3437,   286,   257],
        [ 4081,  1444,  1902, 20935,   654,    11,   543,   925],
        [  198,  7109,  2171,    13,   679,   373,   257,  1263],
        [   11, 12023,    88,   582,   351,  8941,   597,  7393],
 